## Introduction to Online (Incremental) Learning

Online learning, also known as incremental learning, is a type of machine learning where the model updates continuously as new data arrives, rather than being trained all at once on a complete dataset. 

This approach is particularly useful for situations where data flows in continuously or the dataset is too large to fit into memory all at once. Online learning algorithms adjust and improve incrementally as each new piece of data is processed, making them highly adaptable to changes in input data patterns over time.


Challenges in Online Learning

The primary challenges of online learning include:

 - **Handling Non-Stationary Environments:** Data distribution can change over time (concept drift), requiring the model to adapt continuously without human intervention.
 - **Balancing Learning and Performance:** The algorithm must learn from new data while still performing well on the data it has already seen.
 - **Resource Constraints:** Since data is processed continuously, the algorithm must work under constraints of limited memory and processing power.

### Understanding Contextual Bandits

Contextual bandits are a branch of machine learning algorithms that come under the umbrella of reinforcement learning. These algorithms address a specific problem setup where an agent needs to choose actions in a given situation to maximize some notion of cumulative reward. Unlike standard multi-armed bandit problems, which do not take into account the state of the environment or additional context, contextual bandits consider the situation or "context" before making a decision.

Here's the core of how contextual bandits work:
- **Context**: Before making a decision, the algorithm is given some context about the state of the environment. This context can be any relevant information that should influence the decision, such as user profiles, time of day, or the current state of a system.
- **Decision**: Based on the context, the agent chooses an action from a set of possible actions. The action is chosen according to a policy, which is a strategy that the algorithm develops as it learns what actions lead to the best outcomes.
- **Reward**: After taking an action, the agent receives a reward signal. This reward is an indicator of how good the action was in that specific context.

The goal of a contextual bandit algorithm is to learn a policy that maximizes the expected reward over time by balancing the exploration of new actions with the exploitation of actions that are known to yield good results. This setup is particularly useful in scenarios like personalized recommendations or online advertising, where each user's interaction provides a new context and the system must adapt to maximize engagement or revenue.

In [ ]:
# we skipped the installation of vowpalwabbit in requirements.txt
# because it takes a long time to install
# so if you want to run this code, you need to install it manually
# by running the following command:

# ! sudo apt update
# ! sudo apt install -y cmake build-essential libboost-all-dev python3-dev
# ! pip install vowpalwabbit==9.10.0

In [9]:
%%file app_faust_vowpal.py

import faust
import ssl
import vowpalwabbit
from utils import ccloud_lib
from faust_music_events import MusicEvent

# Read the Kafka configuration
kafka_app_config = ccloud_lib.read_ccloud_config("kafka.config")

# Set up SASL credentials
creds = faust.SASLCredentials(
    username=kafka_app_config['sasl.username'],
    password=kafka_app_config['sasl.password'],
    mechanism='PLAIN',
    ssl_context=ssl.create_default_context()
)

# Initialize the Faust app
app = faust.App(
    'music_stream_processor',
    broker=f"kafka://{kafka_app_config['bootstrap.servers']}",
    value_serializer='json',
    store='memory://',
    broker_credentials=creds
)

# Define a Kafka topic with MusicEvent as the value type
topic = app.topic('music_streams', value_type=MusicEvent)
song_plays = app.Table('song_plays', default=int)

# Define a class to maintain state and use Vowpal Wabbit
class MusicRecommender:
    def __init__(self):
        self.model = vowpalwabbit.Workspace("--cb 4", quiet=True)
        self.update_count = 0

    def update_model(self, user_id, item_id, rating):
        example = f"{rating} |u user_{user_id} |i item_{item_id}"
        self.model.learn(example)
        self.update_count += 1

        if self.update_count % 100 == 0:
            self.save_model()

    def predict(self, user_id, item_id):
        example = f"|u user_{user_id} |i item_{item_id}"
        prediction = self.model.predict(example)
        return prediction

    def save_model(self):
        self.model.save("cb.model")

    def load_model(self):
        try:
            self.model = vowpalwabbit.Workspace("--cb 4", quiet=True)
        except FileNotFoundError:
            pass

# Instantiate the recommender
recommender = MusicRecommender()
recommender.load_model()

# Define a stream processor
@app.agent(topic)
async def process(stream):
    async for event in stream:
        if event.page == "NextSong":
            rating = 1
        elif event.page == "Skip":
            rating = 0
        else:
            continue

        user_id = event.userId
        item_id = event.track_id

        recommender.update_model(user_id, item_id, rating)
        song_plays[event.userId] += 1
        print(f'User {event.userId} has listened to {song_plays[event.userId]} songs.')

Overwriting app_faust_vowpal.py
